In [27]:

import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from core.Log import *
from core.plots import *
from core.model_utils import *
from core.CNNmodel import *
from core.benchmarks import MetadataMLP
from core.CardiacCTdataset import DataLoaderFactory
from core.globals import *
import logging
from core.CVsplits import *
from tqdm.notebook import tqdm
setup_loggers()

shape=(64, 64, 64)
pools = ["holdout", "main"]

main_dataset=load_dataset(pool="main")
all_folds_data=get_fold_stats()
DL = DataLoaderFactory(main_dataset, all_folds_data)
OUTER_FOLDS = 4; INNER_FOLDS = 3

#final_experiment = load_from_json(filename="NCV_4_3_folds/OUT_4parameter_grid.json")
#final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
#final_experiment_df = pd.DataFrame(final_experiment)
#final_experiment_df


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
log = logging.getLogger('OUTER_train')
log.info(f"		 	 Model;		ExpID;	OUT;	 HPset;  	 		 Epoch; 		T_loss; 						T_acc;								V_loss; 	 				V_acc;		P; 		LR")
for experiment in final_experiment:
	fold = experiment["OUTER_FOLD"]
	trained = experiment["trained"]
	HPset = experiment["hypers"]["HPset"]
	model_type = experiment["Model"]
	if trained: continue
	if not model_type == "ResNet18": continue
	train_loader, val_loader, _ = DL.create_outer_loaders(fold)
	model, best_model_state = train_RESNET(train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/{model_type}_fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")



	↳ RESNET Experiment 17 | Training model... 
	↳ RESNET Experiment 18 | Training model... 
	↳ RESNET Experiment 19 | Training model... 
	↳ RESNET Experiment 20 | Training model... 


In [24]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",False,False
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",False,False


In [25]:
log = logging.getLogger('OUTER_train')
log.info(f"		 	 Model;		ExpID;	OUT;	 HPset;  	 		 Epoch; 		T_loss; 						T_acc;								V_loss; 	 				V_acc;		P; 		LR")
for experiment in final_experiment:
	trained = experiment["trained"]
	if trained: continue
	model_type = experiment["Model"]
	if not model_type == "SingleView_Sagittal": continue
	fold = experiment["OUTER_FOLD"]
	HPset = experiment["hypers"]["HPset"]
	train_loader, val_loader, _ = DL.create_outer_loaders(fold)
	DR = experiment['hypers']['DR']
	model=SingleViewClassifier(DR)
	model, best_model_state = train_SINGLEVIEW(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/{model_type}_fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")


	↳ Single View Experiment 9 | Training model... 
	↳ Single View Experiment 10 | Training model... 
	↳ Single View Experiment 11 | Training model... 
	↳ Single View Experiment 12 | Training model... 


In [ ]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False


In [29]:
log = logging.getLogger('OUTER_train')
log.info(f"		 	 Model;		ExpID;	OUT;	 HPset;  	 		 Epoch; 		T_loss; 						T_acc;								V_loss; 	 				V_acc;		P; 		LR")
for experiment in final_experiment:
	trained = experiment["trained"]
	if trained: continue
	model_type = experiment["Model"]
	if not model_type == "MLP_META": continue
	fold = experiment["OUTER_FOLD"]
	HPset = experiment["hypers"]["HPset"]
	train_loader, val_loader, _ = DL.create_outer_loaders(fold)
	DR = experiment['hypers']['DR']
	model = MetadataMLP(DR)
	model, best_model_state = train_MLP(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/MLP_META_fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")


	↳ Experiment 1 | Training model... 
	↳ Experiment 2 | Training model... 
	↳ Experiment 3 | Training model... 
	↳ Experiment 4 | Training model... 


## Outer Folds Evaluations

In [ ]:

for experiment in final_experiment:
	print(experiment)
	fold = experiment["Fold"]
	evaluated = experiment["evaluated"]
	HPset = experiment["HPset"]
	#if evaluated: continue
	_, _, test_loader = DL.create_outer_loaders(fold)
	DR = experiment['DR']
	model = MultiViewCNN(DR)
	state_dict = torch.load(f"pth_models/fold_{fold}_HPset_{HPset}.pth")
	model.load_state_dict(state_dict)
	final_loss, all_probabilities, all_labels, all_predictions = EVALUATE_MODEL(model, test_loader, experiment)
	experiment["evaluated"] = True
	save_to_json(final_experiment, filename="training/parameter_grid.json")


Final Model Experiments:   0%|          | 0/8 [00:00<?, ?it/s]

{'ExpID': 1, 'Fold': 0, 'HPset': 6, 'LR': 0.0008, 'WD': 0.0001, 'DR': 0.2, 'TH': 0.5, 'epochs': 50, 'trained': True, 'evaluated': True}


	↳ Experiment 1 | Evaluating model... :   0%|          | 0/4 [00:00<?, ?it/s]

{'ExpID': 2, 'Fold': 1, 'HPset': 6, 'LR': 0.0008, 'WD': 0.0001, 'DR': 0.2, 'TH': 0.5, 'epochs': 50, 'trained': True, 'evaluated': True}


	↳ Experiment 2 | Evaluating model... :   0%|          | 0/4 [00:00<?, ?it/s]

...

	↳ Experiment 6 | Evaluating model... :   0%|          | 0/4 [00:00<?, ?it/s]

{'ExpID': 7, 'Fold': 2, 'HPset': 9, 'LR': 0.0008, 'WD': 1e-06, 'DR': 0.2, 'TH': 0.5, 'epochs': 50, 'trained': True, 'evaluated': True}


	↳ Experiment 7 | Evaluating model... :   0%|          | 0/4 [00:00<?, ?it/s]

{'ExpID': 8, 'Fold': 3, 'HPset': 9, 'LR': 0.0008, 'WD': 1e-06, 'DR': 0.2, 'TH': 0.5, 'epochs': 50, 'trained': True, 'evaluated': True}


	↳ Experiment 8 | Evaluating model... :   0%|          | 0/4 [00:00<?, ?it/s]